# Линейная регрессия

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from model import NeuralNetwork
from utils import *

X = np.linspace(-10, 10, 100).reshape(-1, 1)
y = 2 * X + 1


X = np.array(X)
y = np.array(y)

X_train, y_train, X_test, y_test = X[:int(len(X) * (7 / 10))],  \
                                    y[:int(len(X) * (7 / 10))], \
                                    X[int(len(X) * (7 / 10)):], \
                                    y[int(len(X) * (7 / 10)):]

nn = NeuralNetwork(layer_sizes=[1, 8, 1], task='regression', lr=0.01)
nn.train(X_train, y_train, epochs=1000, batch_size=16)

y_pred = nn.predict(X_test)


print(f"MSE {mse_metric(y_test, y_pred)}")
print(f"RMSE {rmse_metric(y_test, y_pred)}")
print(f"MAE {mae_metric(y_test, y_pred)}")
print(f"MAPE {mape_metric(y_test, y_pred)}")

plt.plot(X_test, y_test, '-o')
plt.plot(X_test, y_pred)
plt.show()

# Нелинейная регрессия

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from model import NeuralNetwork
from utils import *

X = np.linspace(-5, 5, 100).reshape(-1, 1)
y = np.sin(X)
y += np.random.normal(0, 0.05, size=y.shape)

X = np.array(X)
y = np.array(y)

X = z_score_normalizer(X)
y = z_score_normalizer(y)

split_idx = int(len(X) * 0.7)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

nn = NeuralNetwork(layer_sizes=[1, 128, 128, 1], task='regression', lr=0.001, init_method='he', activation='tanh')
nn.train(X_train, y_train, epochs=5000, batch_size=32)

y_pred = nn.predict(X_test)

y_pred = y_pred * np.std(y) + np.mean(y)
y_test = y_test * np.std(y) + np.mean(y)

print(f"MSE {mse_metric(y_test, y_pred)}")
print(f"RMSE {rmse_metric(y_test, y_pred)}")
print(f"MAE {mae_metric(y_test, y_pred)}")
print(f"MAPE {mape_metric(y_test, y_pred)}")

plt.plot(X_test, y_test, '-o')
plt.plot(X_test, y_pred)
plt.show()

# Регрессия (дома)

In [ ]:
import numpy as np
import pandas as pd
from utils import *
from model import NeuralNetwork

train_df = pd.read_csv("./data/train_reg.csv")

y = z_score_normalizer(np.array(train_df["SalePrice"]))
X = train_df.drop(["Id", "SalePrice"], axis=1)
X = X.select_dtypes(include=['float64', 'int64']).fillna(0)
X = z_score_normalizer(X)

X = np.array(X)
y = np.array(y)

split_idx = int(len(X) * 0.7)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

y_train = y_train.reshape(-1, 1)

nn = NeuralNetwork(layer_sizes=[X_train.shape[1], 16, 1], task='regression', lr=0.01)
nn.train(X_train, y_train, epochs=2000, batch_size=16)

y_p = nn.predict(X_test)

print(f"MSE {mse_metric(y_test, y_p)}")
print(f"RMSE {rmse_metric(y_test, y_p)}")
print(f"MAE {mae_metric(y_test, y_p)}")
print(f"MAPE {mape_metric(y_test, y_p)}")

plt.scatter(y_test, y_p, 2)
plt.plot(y_p, y_p, "r")
plt.show()

# Классификация (Титаник)

In [ ]:
import pandas as pd
import numpy as np
from utils import *
from model import NeuralNetwork

data = pd.read_csv("./data/train.csv")

data = data.drop(["PassengerId", "Name", "Age", "Ticket", "Fare", "Cabin"],axis=1)
data = pd.get_dummies(data=data, columns=['Sex', 'Embarked'], prefix=['sex', 'embarked'])

X = data.drop('Survived',axis=1)
y = data['Survived']


split_idx = int(len(X) * 0.7)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]


X_train = X_train.astype(np.float32)
y_train = y_train.astype(np.float32)
X_test = X_test.astype(np.float32)

nn = NeuralNetwork(layer_sizes=[X_train.shape[1], 32, 1], task='binary_classification', lr=0.01)
nn.train(X_train, y_train, epochs=2000, batch_size=8)

y_pred = nn.predict(X_test)

print("Accuracy: {}".format(accuracy(np.array(y_test), y_pred)))
print("Avg harmonical: {}".format(avg_harmonical(np.array(y_test), y_pred)))
print("Recall: {}".format(recall(np.array(y_test), y_pred)))
print("Precision: {}".format(precision(np.array(y_test), y_pred)))

# Мультиклассовая классификация (MNIST)

In [ ]:
import pandas as pd
import numpy as np
from utils import *
from model import NeuralNetwork
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer, StandardScaler

X, y = fetch_openml('mnist_784', version=1, return_X_y=True, parser='auto')
X = X / 255.0
y = LabelBinarizer().fit_transform(y)

X_train, y_train, X_test, y_test = X[:int(len(X) * (7 / 10))],  \
                                    y[:int(len(X) * (7 / 10))], \
                                    X[int(len(X) * (7 / 10)):], \
                                    y[int(len(X) * (7 / 10)):]

X_train = np.array(X_train)
y_train = np.array(y_train)
nn = NeuralNetwork(layer_sizes=[784, 128, 64, 10], task='multiclass_classification', lr=0.01)
nn.train(X_train, y_train, epochs=10, batch_size=128)

y_pred = nn.predict(X_test)

y_test = np.argmax(y_test, axis=1)

print("Accuracy: {}".format(accuracy(np.array(y_test), y_pred)))
print("Avg harmonical: {}".format(avg_harmonical(np.array(y_test), y_pred)))
print("Recall: {}".format(recall(np.array(y_test), y_pred)))
print("Precision: {}".format(precision(np.array(y_test), y_pred)))

---

# PL Линейная задача

In [ ]:
from model import *
from utils import *
import matplotlib.pyplot as plt

train_dataset = LinearDataset(train=True)
test_dataset = LinearDataset(train=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

trainer = pl.Trainer(max_epochs=50)
model = SinModel()
trainer.fit(model, train_loader)

test_results = trainer.test(model, test_loader)[0]
predictions = model.predictions.cpu().numpy()
targets = model.targets.cpu().numpy()

x_values = test_dataset.X.numpy()
y_values = test_dataset.y.numpy()

sorted_indices = np.argsort(x_values[:, 0])
x_sorted = x_values[sorted_indices]
y_true_sorted = y_values[sorted_indices]
y_pred_sorted = predictions[sorted_indices]

plt.figure(figsize=(10, 6))
plt.scatter(x_values, y_values, label='Данные', alpha=0.5)
plt.plot(x_sorted, y_pred_sorted, 'r-', linewidth=2, label='Предсказание')
plt.title('Предсказание нелинейной зависимости')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

# PL Нелинейная задача

In [ ]:
from model import *
from utils import *
import matplotlib.pyplot as plt

train_dataset = SinDataset(train=True)
test_dataset = SinDataset(train=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

trainer = pl.Trainer(max_epochs=100)
model = SinModel()
trainer.fit(model, train_loader)

test_results = trainer.test(model, test_loader)[0]
predictions = model.predictions.cpu().numpy()
targets = model.targets.cpu().numpy()

x_values = test_dataset.X.numpy()
y_values = test_dataset.y.numpy()

sorted_indices = np.argsort(x_values[:, 0])
x_sorted = x_values[sorted_indices]
y_true_sorted = y_values[sorted_indices]
y_pred_sorted = predictions[sorted_indices]

plt.figure(figsize=(10, 6))
plt.scatter(x_values, y_values, label='Данные', alpha=0.5)
plt.plot(x_sorted, y_pred_sorted, 'r-', linewidth=2, label='Предсказание')
plt.title('Предсказание нелинейной зависимости')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

# PL Регрессия (дома)

In [ ]:
from model import *
from utils import *
import matplotlib.pyplot as plt

train_dataset = HousePricesDataset(train=True)
test_dataset = HousePricesDataset(train=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

input_size = len(train_dataset.get_feature_names())
model = HousePriceModel(input_size=input_size)
trainer = pl.Trainer(max_epochs=100)
trainer.fit(model, train_loader)

test_results = trainer.test(model, test_loader)[0]
predictions = model.predictions
targets = model.targets

predictions_orig = test_dataset.inverse_transform_y(predictions)
targets_orig = test_dataset.inverse_transform_y(targets)

plt.figure(figsize=(10, 6))
plt.scatter(targets, predictions, alpha=0.5)
plt.plot([min(targets), max(targets)], 
         [min(targets), max(targets)], 'r--')
plt.xlabel('Истинные значения цены ($)')
plt.ylabel('Предсказанные значения цены ($)')
plt.title('Предсказание цен на дома (исходный масштаб)')
plt.show()

# PL Титаник

In [ ]:
from model import *
from utils import *
import matplotlib.pyplot as plt

dataset = TitanicDataset(train=True)
input_size = dataset.x.shape[1]

train_dataset = TitanicDataset(train=True)
test_dataset = TitanicDataset(train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

model = TitanicModel(input_size=input_size)
trainer = pl.Trainer(max_epochs=20)

trainer.fit(model, train_loader)
test_results = trainer.test(model, test_loader)[0]

# PL MNIST

In [ ]:
from model import *
from utils import *
import matplotlib.pyplot as plt

dataset = MNISTDataset(train=True)

train_dataset = MNISTDataset(train=True)
test_dataset = MNISTDataset(train=False)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128)

model = MNISTModel()
trainer = pl.Trainer(max_epochs=10)

trainer.fit(model, train_loader)
test_results = trainer.test(model, test_loader)[0]

# PL Transfer Learning

In [1]:
from model import *
from utils import *
import matplotlib.pyplot as plt

dataset = MNISTDataset(train=True)

train_dataset = MNISTDataset(train=True)
test_dataset = MNISTDataset(train=False)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128)

model = ResNetTransferLearning()
trainer = pl.Trainer(max_epochs=10)

trainer.fit(model, train_loader)
test_results = trainer.test(model, test_loader)[0]

c:\Users\илья\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\илья\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\илья\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
GPU available: False, used: False
TP

Epoch 9: 100%|██████████| 469/469 [00:54<00:00,  8.65it/s, v_num=68]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 469/469 [00:54<00:00,  8.62it/s, v_num=68]


c:\Users\илья\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 79/79 [00:08<00:00,  9.60it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.7839000225067139     │
│          test_f1          │    0.7839000225067139     │
│         test_loss         │    0.7055922150611877     │
│         test_prec         │    0.7839000225067139     │
│         test_rec          │    0.7839000225067139     │
└───────────────────────────┴───────────────────────────┘